# Stage 3: Inference-Time Grounding Control (CHAIR-only)

This notebook is **separate** from Stage 1/2 and does not modify existing notebooks.

## Goal (Stage 3 only)
Implement a training-free decoding controller that:
1. Monitors visual grounding from Stage 1 hallucination heads.
2. Penalizes likely object-word logits when grounding is weak.
3. Evaluates CHAIR for:
   - Baseline decoding (no inference control)
   - Inference-control decoding

## Stage order
- **Stage 3**: inference-only verification (this notebook).
- **Stage 4**: LoRA + inference integration, then compare Stage 4 vs Stage 2 and Stage 3.

## Inputs expected from previous stages
- `results/final_hallucination_heads.json`
- Stage 1 caches/images under `llava_hallucination_heads/`

In [1]:
# Optional installs for fresh Colab runtimes
!pip -q install -U transformers peft bitsandbytes accelerate pycocotools spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 147.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 16.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import os
import re
import json
import math
import random
import pickle
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from google.colab import drive
from pycocotools.coco import COCO

from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

import spacy
nlp = spacy.load('en_core_web_sm')

drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/llava_hallucination_heads'
COCO_DIR = f'{WORK_DIR}/coco'
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'
HEADS_JSON = f'{WORK_DIR}/results/final_hallucination_heads.json'
CACHE_FILE = f'{WORK_DIR}/cache/screening_state.pkl'
SELECTED_IMGS_JSON = f'{WORK_DIR}/cache/selected_imgs.json'

assert os.path.exists(HEADS_JSON), f'Missing: {HEADS_JSON}'
assert os.path.exists(CACHE_FILE), f'Missing: {CACHE_FILE}'
assert os.path.exists(SELECTED_IMGS_JSON), f'Missing: {SELECTED_IMGS_JSON}'

PROMPT_TEMPLATE = "USER: <image>\nDescribe the image in one sentence.\nASSISTANT:"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [9]:
# Load base model only (Stage 3 is inference-only)
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    device_map={"": 0},
)
model.eval()

print('Loaded base model for Stage 3 inference-only run')

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Loaded base model for Stage 3 inference-only run


In [10]:
# Load Stage 1 artifacts and rebuild eval split (same split style as Stage 2)
with open(HEADS_JSON) as f:
    final_list = json.load(f)

hal_heads_by_layer = defaultdict(set)
for h in final_list:
    hal_heads_by_layer[h['layer']].add(h['head'])

with open(CACHE_FILE, 'rb') as f:
    stage1_state = pickle.load(f)
per_image_cache = stage1_state['per_image_cache']

with open(SELECTED_IMGS_JSON) as f:
    img_meta = json.load(f)
selected_imgs = img_meta['ids']
img_to_gt_objects = {int(k): set(v) for k, v in img_meta['gt_objects'].items()}

ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
cap_ann_path = f'{COCO_DIR}/annotations/captions_val2014.json'
img_dir = f'{COCO_DIR}/val2014_subset'

coco = COCO(ann_path)
_ = COCO(cap_ann_path)

img_meta_coco = coco.loadImgs(selected_imgs)
img_id_to_path = {m['id']: f"{img_dir}/{m['file_name']}" for m in img_meta_coco}

cat_id_to_name = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

# Keep same held-out strategy used in Stage 2
random.seed(42)
eval_images = [img_id for img_id in selected_imgs[-40:] if img_id in img_id_to_path and os.path.exists(img_id_to_path[img_id])]
eval_gt_objects = [img_to_gt_objects[img_id] for img_id in eval_images]

print(f'Hallucination heads: {len(final_list)}')
print(f'Eval images: {len(eval_images)}')

loading annotations into memory...
Done (t=4.14s)
creating index...
index created!
loading annotations into memory...
Done (t=0.33s)
creating index...
index created!
Hallucination heads: 32
Eval images: 40


In [11]:
# CHAIR helpers (mirrors Stage 1/2 vocabulary logic)
COCO_SYNONYMS = {
    'person': ['man','woman','people','boy','girl','child','guy','lady','kid','baby','player','rider','skier','surfer','snowboarder'],
    'car': ['vehicle','automobile','sedan','suv'],
    'dog': ['puppy','dogs'],
    'cat': ['kitten','cats'],
    'tv': ['television','monitor','screen'],
    'couch': ['sofa'],
    'cell phone': ['phone','cellphone','smartphone'],
    'dining table': ['table','desk'],
    'wine glass': ['glass'],
    'bicycle': ['bike'],
    'motorcycle': ['motorbike'],
    'airplane': ['plane','jet'],
    'potted plant': ['plant'],
    'laptop': ['computer'],
    'refrigerator': ['fridge'],
    'truck': ['lorry'],
    'boat': ['ship','sailboat'],
    'fire hydrant': ['hydrant'],
    'hot dog': ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball': ['ball','football','soccer ball','basketball'],
    'baseball bat': ['bat'],
    'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant':'fire hydrant','hotdog':'hot dog','stoplight':'traffic light',
    'bat':'baseball bat','racket':'tennis racket','racquet':'tennis racket',
}

OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())


def normalize_obj_word(w):
    w = w.lower().strip()
    if w in MULTIWORD_ALIASES:
        w = MULTIWORD_ALIASES[w]
    return w


def find_caption_object_words(caption, gt_objects):
    gt_norm = {normalize_obj_word(x) for x in gt_objects}
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)

    doc = nlp(caption)
    obj_words = []
    hall_words = []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2:
            continue
        wn = normalize_obj_word(w)
        if wn in OBJECT_VOCAB:
            obj_words.append(wn)
            if wn not in expanded_gt:
                hall_words.append(wn)
    return obj_words, hall_words


def chair_eval_from_caption_fn(caption_fn, images, gt_objects_list):
    chairs_list, chairi_list = [], []
    examples = []
    for img_id, gt_set in tqdm(list(zip(images, gt_objects_list)), total=len(images), desc='CHAIR'):
        cap = caption_fn(img_id)
        obj_words, hall_words = find_caption_object_words(cap, gt_set)
        chairs_list.append(1 if len(hall_words) > 0 else 0)
        chairi_list.append(len(hall_words) / max(len(obj_words), 1))
        if len(examples) < 5:
            examples.append({'img_id': int(img_id), 'caption': cap, 'hall_words': hall_words})

    return {
        'CHAIRs': float(np.mean(chairs_list)) if chairs_list else 0.0,
        'CHAIRi': float(np.mean(chairi_list)) if chairi_list else 0.0,
        'n': len(chairs_list),
        'examples': examples,
    }

In [12]:
# Build risky layer->heads map and candidate penalty token IDs
risky_heads = defaultdict(set)
for h in final_list:
    risky_heads[int(h['layer'])].add(int(h['head']))


def object_token_ids(tokenizer, object_vocab):
    ids = set()
    for w in sorted(object_vocab):
        for form in [w, ' ' + w]:
            toks = tokenizer.encode(form, add_special_tokens=False)
            if len(toks) == 1:
                ids.add(int(toks[0]))
    return sorted(ids)

PENALTY_TOKEN_IDS = object_token_ids(processor.tokenizer, OBJECT_VOCAB)
print(f'Risky layers: {len(risky_heads)}')
print(f'Risky heads : {sum(len(v) for v in risky_heads.values())}')
print(f'Penalty token ids (single-token object words): {len(PENALTY_TOKEN_IDS)}')

Risky layers: 19
Risky heads : 32
Penalty token ids (single-token object words): 55


In [15]:
# Stage 3 controller: grounding score from risky heads + logit penalty
@torch.no_grad()
def infer_visual_token_span(model_obj, image_path):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    out = model_obj(**inputs, output_attentions=True, return_dict=True)
    attn_len = out.attentions[0].shape[-1]
    text_len = inputs['input_ids'].shape[1]

    # In LLaVA, image features are inserted as extra tokens in the KV sequence.
    # We use the extra prefix length as an approximate visual-token span.
    n_visual = max(0, attn_len - text_len)
    visual_start = 0
    visual_end = n_visual
    return visual_start, visual_end


def grounding_score_from_attentions(attentions, risky_map, visual_start, visual_end, eps=1e-8):
    if visual_end <= visual_start:
        return 0.0

    scores = []
    for layer_idx, heads in risky_map.items():
        if layer_idx >= len(attentions):
            continue
        layer_attn = attentions[layer_idx]  # [B, H, Q, K]
        if layer_attn is None:
            continue

        for h in heads:
            if h >= layer_attn.shape[1]:
                continue
            probs = layer_attn[0, h, -1, :]  # attention over KV for current query token
            vis = probs[visual_start:visual_end]
            vis_mass = vis.sum()

            vis_norm = vis / (vis.sum() + eps)
            vis_entropy = -(vis_norm * (vis_norm + eps).log()).sum()

            score = (vis_mass / (vis_entropy + eps)).item()
            scores.append(score)

    return float(np.mean(scores)) if scores else 0.0


@torch.no_grad()
def generate_caption_baseline(model_obj, image_path, max_new_tokens=80):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    out = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_dict_in_generate=True,
        use_cache=True,
    )
    gen_ids = out.sequences[0, inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(gen_ids, skip_special_tokens=True)


@torch.no_grad()
def generate_caption_with_grounding_control(
    model_obj,
    image_path,
    theta=0.08,
    alpha=8.0,
    max_new_tokens=80,
    eos_token_id=None,
):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    input_ids = inputs['input_ids'].long()
    attention_mask = inputs['attention_mask'].long()
    pixel_values = inputs['pixel_values']

    visual_start, visual_end = infer_visual_token_span(model_obj, image_path)

    past_key_values = None
    generated = []

    for _ in range(max_new_tokens):
        if past_key_values is None:
            out = model_obj(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                use_cache=True,
                past_key_values=None,
                output_attentions=True,
                return_dict=True,
            )
        else:
            out = model_obj(
                input_ids=input_ids[:, -1:],
                attention_mask=attention_mask,
                use_cache=True,
                past_key_values=past_key_values,
                output_attentions=True,
                return_dict=True,
            )

        logits = out.logits[:, -1, :]
        g = grounding_score_from_attentions(out.attentions, risky_heads, visual_start, visual_end)

        if g < theta and PENALTY_TOKEN_IDS:
            penalty = alpha * (g - theta)  # negative when below threshold
            logits[:, PENALTY_TOKEN_IDS] += penalty

        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        token_id = int(next_token.item())
        generated.append(token_id)

        if eos_token_id is not None and token_id == eos_token_id:
            break

        if past_key_values is None:
            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
        else:
            input_ids = next_token
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

        past_key_values = out.past_key_values

    return processor.tokenizer.decode(generated, skip_special_tokens=True)

In [16]:
# Run Stage 3 CHAIR-only evaluation: baseline vs inference control
THETA = 0.08
ALPHA = 8.0
MAX_NEW_TOKENS = 80

def caption_fn_baseline(img_id):
    return generate_caption_baseline(model, img_id_to_path[img_id], max_new_tokens=MAX_NEW_TOKENS)

def caption_fn_inference_control(img_id):
    return generate_caption_with_grounding_control(
        model,
        img_id_to_path[img_id],
        theta=THETA,
        alpha=ALPHA,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

print('=== STAGE 3 EVAL: BASELINE DECODING ===')
stage3_baseline_chair = chair_eval_from_caption_fn(caption_fn_baseline, eval_images, eval_gt_objects)
print({k: v for k, v in stage3_baseline_chair.items() if k != 'examples'})

print('\n=== STAGE 3 EVAL: INFERENCE CONTROL DECODING ===')
stage3_ctrl_chair = chair_eval_from_caption_fn(caption_fn_inference_control, eval_images, eval_gt_objects)
print({k: v for k, v in stage3_ctrl_chair.items() if k != 'examples'})

def fmt_delta(after, before):
    d = after - before
    sign = '+' if d >= 0 else ''
    return f'{sign}{d:.4f}'

print('\n=== CHAIR DELTAS (control - baseline) ===')
print('CHAIRs:', fmt_delta(stage3_ctrl_chair['CHAIRs'], stage3_baseline_chair['CHAIRs']))
print('CHAIRi:', fmt_delta(stage3_ctrl_chair['CHAIRi'], stage3_baseline_chair['CHAIRi']))

=== STAGE 3 EVAL: BASELINE DECODING ===


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'CHAIRs': 0.125, 'CHAIRi': 0.049999999999999996, 'n': 40}

=== STAGE 3 EVAL: INFERENCE CONTROL DECODING ===


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'CHAIRs': 0.075, 'CHAIRi': 0.03333333333333333, 'n': 40}

=== CHAIR DELTAS (control - baseline) ===
CHAIRs: -0.0500
CHAIRi: -0.0167


In [17]:
# Save Stage 3 outputs
stage3_results = {
    'config': {
        'theta': THETA,
        'alpha': ALPHA,
        'max_new_tokens': MAX_NEW_TOKENS,
        'eval_n': len(eval_images),
        'n_risky_heads': int(sum(len(v) for v in risky_heads.values())),
    },
    'baseline_chair': stage3_baseline_chair,
    'inference_control_chair': stage3_ctrl_chair,
}

out_path = f'{WORK_DIR}/results/stage3_inference_eval.json'
with open(out_path, 'w') as f:
    json.dump(stage3_results, f, indent=2)

print('Saved:', out_path)
print('Next: Stage 4 should load this file + Stage 2 results and test LoRA + inference.')

Saved: /content/drive/MyDrive/llava_hallucination_heads/results/stage3_inference_eval.json
Next: Stage 4 should load this file + Stage 2 results and test LoRA + inference.
